# Évaluation quantitative du modèle Llama-3.2-1B fine-tuné (QLoRA) vs modèle de base

Ce notebook est la version interactive du script `evaluate_model.py`.
Il est conçu pour fonctionner sur **GPU** (recommandé avec Unsloth) ou sur **CPU** (fallback via HuggingFace classique).

Métriques calculées :
1. **Perplexity** sur le test set (fine-tuné vs base)
2. **ROUGE-1 / ROUGE-2 / ROUGE-L** (générations vs textes réels)
3. **BERTScore** (générations vs textes réels)
4. **Distinctiveness politique** : classification linéaire sur TF-IDF des textes générés


In [ ]:
!pip install rouge-score bert-score scikit-learn seaborn tqdm pandas matplotlib

## 1. Imports et Détection du matériel (CPU/GPU)

In [ ]:
import json
import os
import re
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import tqdm

# Détection du device (CPU ou GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device détecté : {device}")

if device == "cpu":
    print("\nATTENTION : L'inférence sur CPU sera très lente pour la génération de texte.")
    print("Les optimisations Unsloth et la quantification 4-bit seront automatiquement désactivées.")


## 2. Configuration Globale

In [ ]:
OUT_DIR = Path("eval_results")
OUT_DIR.mkdir(exist_ok=True)

PARTIES_ORDER = [
    "Extreme_Droite", "Droite", "Centre",
    "Socialiste", "Communiste_et_Extreme_Gauche",
    "Ecologiste", "Regionaliste",
]

# 8 profils tests (un par famille politique) - identiques à ceux du script d'origine
CANDIDATS_TEST = [
    {"famille": "Extreme_Droite",              "prenom": "Martial",      "nom": "Delaborde",
     "profession": "Artisan commerçant",        "soutien": "Front national",
     "departement": "Bouches-du-Rhône",         "date": "1988", "tour": "1"},
    {"famille": "Droite",                       "prenom": "Charles-Henri","nom": "de Courcelles",
     "profession": "Chef d'entreprise",         "soutien": "Rassemblement pour la République",
     "departement": "Hauts-de-Seine",           "date": "1988", "tour": "1"},
    {"famille": "Centre",                       "prenom": "François",     "nom": "Lemaire",
     "profession": "Médecin généraliste",       "soutien": "Union pour la démocratie française",
     "departement": "Calvados",                 "date": "1988", "tour": "1"},
    {"famille": "Socialiste",                   "prenom": "Alain",        "nom": "Mignot",
     "profession": "Professeur de lycée",       "soutien": "Parti socialiste",
     "departement": "Nord",                     "date": "1988", "tour": "1"},
    {"famille": "Communiste_et_Extreme_Gauche", "prenom": "Marcel",       "nom": "Roussillon",
     "profession": "Ouvrier métallurgiste",     "soutien": "Parti communiste français",
     "departement": "Seine-Saint-Denis",        "date": "1988", "tour": "1"},
    {"famille": "Ecologiste",                   "prenom": "Brigitte",     "nom": "Valette",
     "profession": "Chercheuse en biologie",    "soutien": "Verts",
     "departement": "Isère",                    "date": "1988", "tour": "1"},
    {"famille": "Regionaliste",                 "prenom": "Yannick",      "nom": "Le Goff",
     "profession": "Agriculteur",               "soutien": "Union démocratique bretonne",
     "departement": "Finistère",                "date": "1988", "tour": "1"},
]


## 3. Fonctions Utilitaires (Helpers)

In [ ]:
def normalize_party(soutien) -> str:
    if pd.isna(soutien) or str(soutien) == "non mentionné":
        return "A_EXCLURE"
    s = str(soutien).lower().split(';')[0].strip()
    if any(x in s for x in ["front national","fn","extrême droite"]):
        return "Extreme_Droite"
    if any(x in s for x in ["communiste","pcf","lutte ouvrière","lcr","psu"]):
        return "Communiste_et_Extreme_Gauche"
    if any(x in s for x in ["écolog","ecolog","vert","biosphère"]) and "chasse" not in s:
        return "Ecologiste"
    if any(x in s for x in ["socialiste","ps","mrg","radicaux de gauche"]):
        return "Socialiste"
    if any(x in s for x in ["rpr","gaulliste","parti républicain","droite"]):
        return "Droite"
    if any(x in s for x in ["udf","centre","cds","parti radical"]):
        return "Centre"
    if any(x in s for x in ["corse","breton","catalan","indépendantiste"]):
        return "Regionaliste"
    if any(x in s for x in ["sans étiquette","indépendant"]):
        return "Sans_Etiquette"
    return "A_EXCLURE"

def build_prompt(row: dict, tokenizer) -> str:
    system_msg = (
        "Tu es un expert en rhétorique politique française et un archiviste spécialisé "
        "dans l'histoire de la Ve République. Ta tâche est de rédiger une profession de "
        "foi électorale historique et convaincante."
    )
    user_msg = (
        "Rédige la profession de foi à partir des caractéristiques suivantes :\n"
        f"- Candidat : {row.get('prenom', '')} {row.get('nom', '')}\n"
        f"- Profession : {row.get('profession', '')}\n"
        f"- Soutien politique : {row.get('soutien', '')}\n"
        f"- Élection : Élections législatives, Tour {row.get('tour', '')}\n"
        f"- Date : {row.get('date', '')}\n"
        f"- Département : {row.get('departement', '')}"
    )
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def generate_text(model, tokenizer, prompt: str, max_new_tokens=512, device="cuda") -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            top_p=0.9,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # On ne décode que les tokens générés
    generated = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


## 4. Fonctions d'Évaluation (Perplexité, ROUGE, BERTScore, Distinctivité)

In [ ]:
def compute_perplexity(model, tokenizer, texts: list, device="cuda", max_length=1024, batch_size=4) -> float:
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    for i in tqdm(range(0, len(texts), batch_size), desc="Perplexity"):
        batch_texts = texts[i:i+batch_size]
        enc = tokenizer(
            batch_texts, return_tensors="pt", padding=True, 
            truncation=True, max_length=max_length
        ).to(device)
        
        labels = enc["input_ids"].clone()
        labels[enc["attention_mask"] == 0] = -100  # Ignore padding
        
        with torch.no_grad():
            out = model(**enc, labels=labels)
            
        n_tokens = (labels != -100).sum().item()
        total_nll += out.loss.item() * n_tokens
        total_tokens += n_tokens

    mean_nll = total_nll / total_tokens if total_tokens > 0 else float("inf")
    return float(np.exp(mean_nll))

def compute_rouge(hypotheses: list, references: list) -> dict:
    try:
        from rouge_score import rouge_scorer
    except ImportError:
        print("⚠ rouge_score non installé. Utilisez `pip install rouge-score`")
        return {}

    scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=False)
    results = defaultdict(list)
    for hyp, ref in zip(hypotheses, references):
        scores = scorer.score(ref, hyp)
        for k, v in scores.items():
            results[k].append(v.fmeasure)
    return {k: float(np.mean(v)) for k, v in results.items()}

def compute_bertscore(hypotheses: list, references: list, lang="fr") -> dict:
    try:
        from bert_score import score as bert_score_fn
    except ImportError:
        print("⚠ bert_score non installé. Utilisez `pip install bert-score`")
        return {}

    P, R, F1 = bert_score_fn(hypotheses, references, lang=lang, verbose=False)
    return {
        "precision": float(P.mean()),
        "recall":    float(R.mean()),
        "f1":        float(F1.mean()),
        "f1_per_doc": F1.tolist(),
    }

def political_distinctiveness(generated_texts: list, generated_labels: list, real_texts: list, real_labels: list) -> dict:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
    import seaborn as sns

    vec = TfidfVectorizer(max_features=5000, sublinear_tf=True, token_pattern=r"(?u)\b[a-zà-ÿ]{3,}\b")
    X_train = vec.fit_transform([t.lower() for t in real_texts])
    clf = LogisticRegression(max_iter=500, C=1.0, random_state=42)
    clf.fit(X_train, real_labels)

    X_gen = vec.transform([t.lower() for t in generated_texts])
    preds = clf.predict(X_gen)
    acc   = accuracy_score(generated_labels, preds)

    print(f"\n=== Distinctiveness politique ===")
    print(f"Accuracy (prédiction étiquette sur générations) : {acc:.3f}")
    print(classification_report(generated_labels, preds, zero_division=0))

    labels_unique = sorted(set(generated_labels))
    cm = confusion_matrix(generated_labels, preds, labels=labels_unique)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_unique, yticklabels=labels_unique, ax=ax)
    ax.set_title("Matrice de confusion : familles politiques (générations)", fontsize=12)
    ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "political_classifier_confusion.png", dpi=150)
    plt.show()

    return {"accuracy": acc, "predictions": preds.tolist()}


## 5. Fonctions de tracé graphique (Plots)

In [ ]:
def plot_perplexity_comparison(ppl_base: float, ppl_finetuned: float):
    fig, ax = plt.subplots(figsize=(5, 4))
    models  = ["Llama-3.2-1B\n(base)", "Llama-3.2-1B\n(fine-tuné)"]
    values  = [ppl_base, ppl_finetuned]
    colors  = ["#999999", "#2196F3"]
    bars = ax.bar(models, values, color=colors, edgecolor="white", width=0.4)
    ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=11, fontweight="bold")
    ax.set_title("Perplexité sur le test set Archelec", fontsize=12)
    ax.set_ylabel("Perplexité (↓ meilleure)")
    ax.set_ylim(0, max(values) * 1.25)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "perplexity_comparison.png", dpi=150)
    plt.show()

def plot_rouge_scores(rouge_base: dict, rouge_ft: dict):
    if not rouge_base or not rouge_ft:
        return
    metrics = list(rouge_base.keys())
    x = np.arange(len(metrics))
    w = 0.35
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - w/2, [rouge_base[m] for m in metrics], w, label="Base", color="#999", edgecolor="white")
    ax.bar(x + w/2, [rouge_ft[m]   for m in metrics], w, label="Fine-tuné", color="#2196F3", edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.set_ylabel("F1 score")
    ax.set_title("Scores ROUGE : base vs fine-tuné (test set)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "rouge_scores.png", dpi=150)
    plt.show()

def plot_bertscore_distribution(f1_base: list, f1_ft: list):
    if not f1_base or not f1_ft:
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(f1_base, bins=30, alpha=0.6, label="Base", color="#999")
    ax.hist(f1_ft,   bins=30, alpha=0.6, label="Fine-tuné", color="#2196F3")
    ax.axvline(np.mean(f1_base), color="gray", lw=2, ls="--", label=f"Moy. base {np.mean(f1_base):.3f}")
    ax.axvline(np.mean(f1_ft),   color="#0D47A1", lw=2, ls="--", label=f"Moy. FT {np.mean(f1_ft):.3f}")
    ax.set_title("Distribution BERTScore F1 : base vs fine-tuné")
    ax.set_xlabel("BERTScore F1")
    ax.set_ylabel("Fréquence")
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "bertscore_distribution.png", dpi=150)
    plt.show()


## 6. Chargement des Modèles
Le code s'adapte automatiquement selon que vous utilisez un GPU ou un CPU.

In [ ]:
model_id = "fdechamps/Llama-1B-Archelec-20260409-1742" # Changez ce lien vers votre propre endpoint si nécessaire
base_id = "meta-llama/Llama-3.2-1B-Instruct"

print("Chargement des modèles...")

if device == "cuda":
    from unsloth import FastLanguageModel
    
    print("Configuration GPU détectée : Utilisation de Unsloth et quantification 4-bit.")
    # Modèle fine-tuné
    model_ft, tokenizer = FastLanguageModel.from_pretrained(
        model_id, max_seq_length=2048, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model_ft)
    
    # Modèle de base
    model_base, _ = FastLanguageModel.from_pretrained(
        base_id, max_seq_length=2048, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model_base)
    
else:
    print("Configuration CPU détectée : Fallback sur HuggingFace (PeftModel) classique.")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    
    tokenizer = AutoTokenizer.from_pretrained(base_id)
    
    print("Chargement du modèle de base en mémoire...")
    model_base = AutoModelForCausalLM.from_pretrained(
        base_id, device_map="cpu", torch_dtype=torch.float32
    )
    
    print("Application des poids LoRA...")
    try:
        model_ft = PeftModel.from_pretrained(model_base, model_id)
    except Exception as e:
        print(f"Erreur au chargement des poids LoRA : {e}")
        model_ft = model_base  # Fallback final en cas d'erreur


## 7. Chargement des données de test

In [ ]:
from datasets import load_dataset

data_dir = "data"
n_test = 100

PATTERNS = [
    f"{data_dir}/1981/legislatives/*PF*.txt",
    f"{data_dir}/1988/legislatives/*PF*.txt",
    f"{data_dir}/1993/legislatives/*PF*.txt",
]

files = [p for pat in PATTERNS for p in sorted(Path().glob(pat))]
print("Chargement du dataset...")

dataset = load_dataset("text", data_files=PATTERNS, split="train", sample_by="document")
meta = pd.read_csv(f"{data_dir}/archelect_search.csv")
COLS = ["id","titulaire-prenom","titulaire-nom","titulaire-profession",
        "titulaire-soutien","contexte-tour","date","departement-nom"]
meta_dict = meta[[c for c in COLS if c in meta.columns]].set_index("id").to_dict("index")

def add_meta(ex, idx):
    p = files[idx]
    ex["id"] = p.stem
    m = meta_dict.get(p.stem, {})
    for k, col in [("prenom","titulaire-prenom"),("nom","titulaire-nom"),
                    ("profession","titulaire-profession"),("soutien","titulaire-soutien"),
                    ("tour","contexte-tour"),("date","date"),("departement","departement-nom")]:
        ex[k] = m.get(col, "non mentionné")
    return ex

dataset = dataset.map(add_meta, with_indices=True)
df = dataset.to_pandas()
df["famille"] = df["soutien"].apply(normalize_party)
df = df[df["famille"] != "A_EXCLURE"]

test_df = df.sample(frac=0.1, random_state=42).head(n_test)
print(f"\n{len(test_df)} documents dans le test set d'évaluation.")


## 8. Lancement des Évaluations

In [ ]:
max_gen = 512

# --- A. PERPLEXITÉ ---
print("\n=== A. Perplexité ===")
test_texts = test_df["text"].tolist()
ppl_ft   = compute_perplexity(model_ft,   tokenizer, test_texts, device)
ppl_base = compute_perplexity(model_base, tokenizer, test_texts, device)

print(f"  Perplexité base     : {ppl_base:.2f}")
print(f"  Perplexité fine-tuné: {ppl_ft:.2f}")
plot_perplexity_comparison(ppl_base, ppl_ft)

# --- B. ROUGE & BERTSCORE ---
print("\n=== B. ROUGE & BERTScore ===")
print("Génération des réponses sur le test set (attention, très chronophage sur CPU)...")
hypotheses_ft, hypotheses_base, references = [], [], []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    prompt = build_prompt(row.to_dict(), tokenizer)
    hypotheses_ft.append(generate_text(model_ft, tokenizer, prompt, max_gen, device))
    hypotheses_base.append(generate_text(model_base, tokenizer, prompt, max_gen, device))
    references.append(row["text"])

rouge_ft   = compute_rouge(hypotheses_ft,   references)
rouge_base = compute_rouge(hypotheses_base, references)
print(f"  ROUGE fine-tuné : {rouge_ft}")
print(f"  ROUGE base      : {rouge_base}")
plot_rouge_scores(rouge_base, rouge_ft)

bs_ft   = compute_bertscore(hypotheses_ft,   references)
bs_base = compute_bertscore(hypotheses_base, references)
print(f"  BERTScore F1 fine-tuné : {bs_ft.get('f1', 'N/A'):.3f}")
print(f"  BERTScore F1 base      : {bs_base.get('f1', 'N/A'):.3f}")
plot_bertscore_distribution(bs_base.get("f1_per_doc",[]), bs_ft.get("f1_per_doc",[]))

# --- C. DISTINCTIVITÉ POLITIQUE ---
print("\n=== C. Distinctivité politique ===")
gen_texts, gen_labels = [], []
# On génère plusieurs fois chaque profil test pour alimenter la régression logistique
for c in CANDIDATS_TEST * 3:
    prompt = build_prompt(c, tokenizer)
    gen_texts.append(generate_text(model_ft, tokenizer, prompt, max_gen, device))
    gen_labels.append(c["famille"])

distinctiveness = political_distinctiveness(
    gen_texts, gen_labels,
    test_df["text"].tolist(), test_df["famille"].tolist()
)

# --- D. SAUVEGARDE JSON ---
print("\n=== D. Sauvegarde des résultats ===")
summary = {
    "perplexity_base":       ppl_base,
    "perplexity_finetuned":  ppl_ft,
    "rouge_base":            rouge_base,
    "rouge_finetuned":       rouge_ft,
    "bertscore_base":        {k: v for k, v in bs_base.items() if k != "f1_per_doc"},
    "bertscore_finetuned":   {k: v for k, v in bs_ft.items()   if k != "f1_per_doc"},
    "political_accuracy":    distinctiveness.get("accuracy"),
}
(OUT_DIR / "eval_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"\n✓ Résultats sauvegardés dans {OUT_DIR}/eval_summary.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))
